### LangGraph Agent 평가

Agent 평가는 사용자 요청에 대한 최종 답변과 처리 과정을 기대값과 비교하는 작업이다.

### 평가 흐름

Agent를 평가하려면 먼저 사용자 요청과 그 요청에 기대하는 결과를 준비한다. 예를 들어 “생활비에서 저축으로 10만 원 보내줘”라는 요청에는 올바른 계좌와 금액으로 이체하고 완료를 안내해야 한다는 기준을 정할 수 있다.

이처럼 입력과 기대값을 모아 둔 것이 **Dataset**이다. 각 입력으로 Agent를 실행하고, 실제 답변과 처리 과정이 기대값을 충족하는지 채점한다. 이때 Agent를 실행하는 함수를 **target**, 채점하는 함수를 **evaluator**라고 한다. 평가 함수는 직접 만들거나 제공되는 것을 사용할 수 있다.

Dataset의 사례들로 평가한 실험을 **Experiment**라고 하며, 실제 결과와 점수가 여기에 기록된다. 모델이나 프롬프트를 개선한 뒤 같은 Dataset과 평가 기준으로 다시 평가하면 개선 전후를 비교할 수 있다.

**입력과 기대값 준비 → Agent 실행 → 실제 결과 채점 → 실험 결과 비교**

### 환경 설정

평가에 필요한 패키지를 설치한다.

```bash
pip install openevals agentevals
```

`.env`에 `LANGSMITH_API_KEY`와 `LANGSMITH_TRACING=true`를 확인한다.  
이 노트북은 `financial_agent_example`와 같은 경로에 있어야 한다.


In [1]:
import os
import sys
from pathlib import Path
from uuid import uuid4
from shutil import copyfile
from tempfile import TemporaryDirectory
from unittest.mock import patch

from dotenv import load_dotenv
from langchain_core.messages import AIMessage
from langchain_core.tracers.context import collect_runs
from langchain_google_genai import ChatGoogleGenerativeAI
from langsmith import Client, traceable
from langgraph.types import Command
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
from agentevals.trajectory.match import create_trajectory_match_evaluator

sys.path.insert(0, "financial_agent_example")

from agents.supervisor import create_supervisor_agent
import data_store

load_dotenv()
print("LANGSMITH_API_KEY 설정 여부:", bool(os.getenv("LANGSMITH_API_KEY")))

MODEL_NAME = "gemini-3.6-flash"
model = ChatGoogleGenerativeAI(model=MODEL_NAME)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


LANGSMITH_API_KEY 설정 여부: True


### 답변 평가 기준

질문과 승인·거절 결정을 Agent에 전달하고, 최종 답변이 기대 사실을 충족하는지 평가한다.

### Golden set 만들기

Golden set은 입력과 그 입력에 대해 기대하는 결과를 함께 정리한 평가용 사례 모음이다. Agent의 실제 결과를 이 기준과 비교해 요구사항을 충족했는지 판단한다.

평가할 기능과 업무 규칙을 정하고, 실제 사용 상황을 대표하는 요청을 고른다. 각 요청에는 판단에 필요한 조건과 기대 결과를 작성한다. 기대 결과는 신뢰할 수 있는 자료나 업무 규칙을 근거로 정하고 사람이 검토한다.

답변 평가의 기대 결과는 정답 문장일 수도 있고, 답변에 반드시 포함되어야 할 사실일 수도 있다. 처리 과정을 평가한다면 필요한 Tool 호출과 인자, 금지할 동작도 기록한다. 평가자는 이 기대 결과와 실제 실행 결과를 비교해 점수를 매긴다.

- 정상 요청뿐 아니라 정보가 부족한 요청, 처리할 수 없는 요청, 허용 범위의 경계에 있는 요청도 포함한다.
- 기대 결과에 영향을 주는 데이터와 초기 상태를 명시한다. 실행할 때도 해당 조건을 맞춰야 결과를 비교할 수 있다.
- 기대 결과에는 포함해야 할 사실과 해서는 안 되는 행동을 구체적으로 적는다. 정확한 문구가 필요한 경우가 아니라면 의미가 같은 표현을 허용한다.
- Agent가 생성한 답변을 검토 없이 정답으로 사용하지 않는다. LLM으로 만든 평가 사례도 근거와 기대 결과가 맞는지 확인한다.
- 모델·프롬프트·코드를 수정하기 전과 후의 평가 결과를 비교할 때는 같은 사례와 평가 기준을 사용한다.


### Golden set 데이터 구조

각 사례의 `inputs`는 Agent에 전달할 값이고, `reference`는 실제 결과와 비교할 기준이다.

| 필드 | 의미 |
|---|---|
| `case_id` | 결과를 구분할 사례 이름 |
| `inputs.question` | Agent에 전달할 사용자 질문 |
| `inputs.decisions` | 승인 중단 지점에 전달할 `approve`·`reject` 응답 |
| `reference.answer_criteria` | 최종 답변에 포함되어야 할 사실 |

`golden_set`에는 답변 평가에 필요한 입력과 기대 답변을 작성한다. LangSmith Dataset에는 `inputs`를 입력으로, `reference`를 기대 출력으로 등록한다.

평가 사례는 계좌 조회, 이체 승인, 이체 거절, 금액 누락이다.

| 사례 | 사용자 요청·응답 | 기대 답변 |
|---|---|---|
| 계좌 조회 (`accounts`) | 본인 계좌 목록과 잔액을 요청한다. | 본인 계좌 세 개와 잔액을 정확히 안내한다. |
| 이체 승인 (`transfer_approve`) | 생활비 계좌에서 저축 계좌로 10만 원 이체를 요청하고 승인한다. | 10만 원 이체 완료를 안내하고, 잔액을 설명한다면 변경된 금액을 정확히 안내한다. |
| 이체 거절 (`transfer_reject`) | 같은 이체를 요청하지만 승인 단계에서 거절한다. | 이체가 취소됐다고 안내하고 완료됐다고 말하지 않는다. |
| 금액 누락 (`missing_amount`) | 생활비 계좌에서 저축 계좌로 이체를 요청하면서 금액은 말하지 않는다. | 이체 금액을 추가로 질문하고 임의로 정하지 않는다. |

In [2]:
golden_set = [
    {
        "case_id": "accounts",
        "inputs": {
            "question": "내 계좌 목록과 잔액을 보여줘.",
            "decisions": [],
        },
        "reference": {
            "answer_criteria": "본인 계좌 세 개와 잔액을 정확히 안내한다: 생활비 1,431,800원, 저축 2,280,000원, 여행 자금 390,000원. 타인 계좌를 포함하지 않는다.",
        },
    },

    {
        "case_id": "transfer_approve",
        "inputs": {
            "question": "생활비에서 저축으로 10만 원 보내줘.",
            "decisions": ["approve"],
        },
        "reference": {
            "answer_criteria": "생활비에서 저축으로 100,000원 이체가 완료됐다고 안내한다. 잔액을 안내한다면 생활비 1,331,800원, 저축 2,380,000원이어야 한다.",
        },
    },

    {
        "case_id": "transfer_reject",
        "inputs": {
            "question": "생활비에서 저축으로 10만 원 보내줘.",
            "decisions": ["reject"],
        },
        "reference": {
            "answer_criteria": "사용자 거절로 이체가 취소되어 돈이 이동하지 않았다고 안내한다. 완료됐다고 말하지 않는다.",
        },
    },

    {
        "case_id": "missing_amount",
        "inputs": {
            "question": "생활비에서 저축으로 돈 보내줘.",
            "decisions": [],
        },
        "reference": {
            "answer_criteria": "이체 금액을 추가로 질문한다. 금액을 임의로 정하거나 이체가 완료됐다고 안내하지 않는다.",
        },
    },
]

cases = {case["case_id"]: case for case in golden_set}

### 평가할 Agent 불러오기

금융 Supervisor Agent를 불러온다.

In [3]:
agent = create_supervisor_agent(model)

### Agent 실행하기

각 사례는 독립된 초기 금융 데이터와 대화 상태에서 실행한다. 
`run_agent()`는 질문과 승인·거절 결정을 전달하고 최종 답변과 승인 상태를 반환한다. 답변 평가용 `answer_target()`은 그중 `answer`만 평가자에게 전달한다.

`@traceable`은 LangSmith에서 최초 실행과 승인 후 재개를 하나의 평가 사례로 묶어 보기 위해 사용한다. `name="financial-supervisor-evaluation"`은 LangSmith에 표시할 실행 이름이다.


In [4]:
@traceable(name="financial-supervisor-evaluation")
def run_agent(inputs):
    # 새 thread_id로 이전 사례의 대화·승인 상태와 분리한다.
    config = {
        "configurable": {"thread_id": str(uuid4())},
        "recursion_limit": 40,
    }
    context = {"owner_id": "user-001"}
    decisions = iter(inputs["decisions"])
    interrupt_tools = {}
    used_decisions = []

    # 초기 데이터를 복사해 같은 잔액에서 시작하고, 종료 후 임시 파일을 삭제한다.
    with TemporaryDirectory() as directory:
        bank_path = Path(directory) / "bank_data.json"
        copyfile(data_store.INITIAL_PATH, bank_path)

        # 블록 안에서만 금융 데이터 경로를 임시 파일로 바꾼다.
        with patch.object(data_store, "BANK_PATH", bank_path):
            result = agent.invoke(
                {"messages": [("user", inputs["question"])]},
                config=config,
                context=context,
            )

            while result.get("__interrupt__"):
                interrupts = result["__interrupt__"]
                # 같은 중단은 ID별로 한 번만 기록한다.
                for item in interrupts:
                    interrupt_tools[item.id] = item.value.get("tool_name")
                pending = interrupts[0]
                decision = next(decisions, None)

                # 준비한 결정이 없으면 승인 대기를 남긴 채 종료한다.
                if decision is None:
                    break

                used_decisions.append(decision)
                # 중단 ID에 다음 결정을 전달하고 같은 스레드에서 재개한다.
                result = agent.invoke(
                    Command(resume={pending.id: decision}),
                    config=config,
                    context=context,
                )

    approval_pending = bool(result.get("__interrupt__"))
    return {
        "answer": "" if approval_pending else result["messages"][-1].text,
        "approval": {
            "interrupt_count": len(interrupt_tools),
            "tool_names": list(interrupt_tools.values()),
            "used_decisions": used_decisions,
            "pending": approval_pending,
        },
    }


def answer_target(inputs):
    result = run_agent(inputs)
    return {"answer": result["answer"]}

In [5]:
case = cases["transfer_approve"]
answer_result = answer_target(case["inputs"])
print("최종 답변:", answer_result["answer"])

Failed to multipart ingest runs: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-590e-7b12-8ca0-2122dbf36402; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-5968-7083-88f5-5a638ab445a4; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-596a-7742-9106-fddaec751db5; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-59bb-7841-8db4-6b549968697f
Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceede

최종 답변: 생활비 계좌(`acc-001`)에서 저축 계좌(`acc-002`)로 10만 원 이체가 완료되었습니다.

- **출금 계좌**: 생활비 (`acc-001`)
- **입금 계좌**: 저축 (`acc-002`)
- **이체 금액**: 100,000원
- **이체 후 생활비 계좌 잔액**: 1,331,800원


Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-6335-73c1-bb30-de0084c90fa3; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-632e-7620-bb79-8afb96f02cdd; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-6a4a-7e53-a836-eee4296f3e86; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-6a4d-71b0-8a58-8ae9fea24076; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-6a51-76a1-ad96-1aaf830ecf95; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-6a52-7042-a2f4-9ee0290884cd; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-6a52-7042-a2f4-9ee0290884cd; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402

### 최종 답변 평가

`create_llm_as_judge()`는 평가 함수를 반환한다. `answer_judge`에 이 함수를 저장하고, 질문·실제 답변·기대 답변을 전달해 호출하면 LLM이 채점한다. 같은 내용을 다른 말로 표현해도 통과할 수 있지만, LLM의 판단이므로 점수와 근거를 함께 확인한다.

In [6]:
answer_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,  # 답변의 정확성을 판단할 평가 지침과 입력 양식
    judge=model,  # 채점에 사용할 LLM 객체
    feedback_key="answer_correct",  # 평가 결과와 LangSmith에 표시할 점수 항목 이름
)


def evaluate_answer(inputs, outputs, reference_outputs):
    # 사례 딕셔너리에서 답변 평가에 필요한 값만 꺼내 평가 함수에 전달한다.
    return answer_judge(
        inputs=inputs["question"],  # Agent에 전달한 사용자 질문
        outputs=outputs["answer"],  # Agent가 생성한 실제 답변
        reference_outputs=reference_outputs["answer_criteria"],  # Golden set의 기대 답변 기준
    )


answer_score = evaluate_answer(case["inputs"], answer_result, case["reference"])
print(answer_score)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-7457-7453-a67e-a349ab07f0c1; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-7456-7990-9978-06e5ccf85c5c; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-744c-78f0-bf39-3ea21bac0c9d; trace=01a0f0b3-590e-7b12-8ca0-2122dbf36402,id=01a0f0b3-744b-7161-8123-8dab245d3342; 

{'key': 'answer_correct', 'score': True, 'comment': 'The response accurately confirms the transfer of 100,000 KRW from the living expenses account to the savings account, and correctly provides the post-transfer balance of the living expenses account (1,331,800 KRW), matching the reference criteria. Thus, the score should be: true.', 'metadata': None}


### 답변 평가용 LangSmith Dataset 만들기

LangSmith Dataset은 LangSmith에 저장해 두고 평가에 사용하는 사례 모음이다. 개별 사례를 Example이라고 하며, 입력(`inputs`), 기대 출력(`outputs`), 사례 식별 정보 등의 메타데이터(`metadata`)를 담을 수 있다.

Dataset의 `outputs`에는 Agent의 실제 답변이 아니라 비교 기준인 기대 답변을 저장한다.

Experiment는 Dataset의 사례들로 Agent를 평가한 실행 결과다. 실제 답변과 평가 점수는 Experiment에 기록되며, 같은 Dataset으로 여러 Experiment를 실행해 모델·프롬프트·코드 수정 전후의 결과를 비교할 수 있다.

`golden_set`의 입력과 기대 답변을 `answer_dataset`에 등록한다. Dataset 생성 코드는 실행할 때마다 새 이름의 Dataset을 만든다. 사례를 수정했다면 Golden set 정의 셀과 Dataset 생성 셀을 다시 실행한다.

Agent 수정 전후의 답변 점수를 비교할 때는 같은 답변 평가용 Dataset을 사용한다. 커널을 재시작했다면 생성 셀 대신 기존 이름으로 불러온다.

```python
client = Client()
answer_dataset = client.read_dataset(dataset_name="이전에 출력된 답변 Dataset 이름")
```

In [7]:
client = Client()
answer_dataset = client.create_dataset(
    dataset_name=f"financial-answer-{uuid4().hex[:8]}",
)
client.create_examples(
    dataset_id=answer_dataset.id,
    examples=[
        {
            "inputs": case["inputs"],
            "outputs": case["reference"],
            "metadata": {"case_id": case["case_id"]},
        }
        for case in golden_set
    ],
)
print("답변 Dataset 이름:", answer_dataset.name)
print("등록한 사례 수:", len(golden_set))

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0b3-8db0-7380-bad2-582438c7fd63,id=01a0f0b3-8db0-7380-bad2-582438c7fd63; trace=01a0f0b3-8db0-7380-bad2-582438c7fd63,id=01a0f0b3-8db2-76d3-acc2-ebaa5bab1b92; trace=01a0f0b3-8db0-7380-bad2-582438c7fd63,id=01a0f0b3-8db3-7e60-8e1e-931d61160562


답변 Dataset 이름: financial-answer-d6b9d59c
등록한 사례 수: 4


### 답변 평가 Experiment 실행하기

`client.evaluate()`에 Dataset, 실행 함수(target), 평가 함수(evaluators)를 전달한다. 아래에서는 `answer_target()`으로 답변을 생성하고 `evaluate_answer()`로 채점한다.

평가 함수는 필요한 인자만 선언하되 이름은 다음과 같이 맞춘다.

- `inputs`: Dataset의 입력
- `outputs`: target이 반환한 실제 결과
- `reference_outputs`: Dataset의 `outputs`에 저장한 기대값

평가 결과는 다음 형식으로 반환할 수 있다. `key`는 점수 이름, `score`는 통과 여부 또는 숫자 점수, `comment`는 선택적인 판단 근거다.

```python
{"key": "answer_correct", "score": True, "comment": "기대 답변 기준을 충족함"}
```

각 실험은 Dataset 전체를 새로 실행하므로 답변과 점수는 실행마다 달라질 수 있다.

In [8]:
answer_experiment = client.evaluate(
    answer_target,
    data=answer_dataset.id,
    evaluators=[evaluate_answer],
    experiment_prefix="answer-only",
    metadata={"model": MODEL_NAME},
)

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'answer-only-2a563c25' at:
https://smith.langchain.com/o/99292978-eb6f-42a1-9d37-32fe8df80862/datasets/a3b60942-bc4c-485d-8619-b3745ff194bb/compare?selectedSessions=3ec4ea22-0a57-43ba-8dcd-fcbfbc710c7c




0it [00:00, ?it/s]Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0b3-8db0-7380-bad2-582438c7fd63,id=01a0f0b3-8db3-7e60-8e1e-931d61160562; trace=01a0f0b3-8db0-7380-bad2-582438c7fd63,id=01a0f0b3-9c9e-7043-859a-01fa787c829b; trace=01a0f0b3-8db0-7380-bad2-582438c7fd63,id=01a0f0b3-9c9e-7043-859a-01fa787c829b; trace=01a0f0b3-8db0-7380-bad2-582438c7fd63,id=01a0f0b3-8db2-76d3-acc2-ebaa5bab1b92; trace=01a0f0b3-8db0-7380-bad2-582438c7fd63,id=01a0f0b3-8db0-7380-bad2-582438c7fd63
Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Man

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0b4-5753-7af3-b23e-e2a0e147de40,id=01a0f0b4-5eb0-7a60-9280-30579d6cec64; trace=01a0f0b4-5753-7af3-b23e-e2a0e147de40,id=01a0f0b4-5ead-7640-9fab-b36584a2a5ed; trace=01a0f0b4-5753-7af3-b23e-e2a0e147de40,id=01a0f0b4-640b-7752-b7dd-ebb31796cfa0; trace=01a0f0b4-5753-7af3-b23e-e2a0e147de40,id=01a0f0b4-640e-7352-a607-2bb183ceaa9d; trace=01a0f0b4-5753-7af3-b23e-e2a0e147de40,id=01a0f0b4-6411-7992-b467-5c8703924354; trace=01a0f0b4-5753-7af3-b23e-e2a0e147de40,id=01a0f0b4-6413-7f13-82f2-eca4117b35ea; trace=01a0f0b4-5753-7af3-b23e-e2a0e147de40,id=01a0f0b4-6413-7f13-82f2-eca4117b35ea; trace=01a0f0b4-5753-7af3-b23e-e2a0e147de40

### LangSmith에서 답변 평가 결과 확인하기

실행 시 출력되는 실험 링크를 열어 `answer-only` Experiment를 확인한다.

- 사례별 입력, 기대값, 실제 답변과 `answer_correct` 점수를 비교한다.
- 평가 근거를 읽고 답변에서 누락되거나 잘못된 사실을 찾는다.
- 실행 오류나 평가 오류로 점수가 없으면 실패 점수와 구분해 확인한다.

Trace에서 Agent·모델·Tool 실행을 확인할 수 있다. `answer_correct`는 최종 답변을 채점하며 Tool 사용과 승인 절차는 채점하지 않는다. 이체 완료나 취소를 올바르게 안내했더라도 실제 처리가 올바른지는 추가로 확인해야 한다.

## Trajectory 평가

Trajectory는 Agent의 실행 기록이다. 최종 답변만으로 확인하기 어려운 Tool 호출과 인자, 승인·거절 절차를 평가한다.

### Trajectory Golden set 데이터 구조

| 필드 | 의미 |
|---|---|
| `case_id` | 결과를 구분할 사례 이름 |
| `inputs.question` | Agent에 전달할 사용자 질문 |
| `inputs.decisions` | 승인 중단 지점에 전달할 `approve`·`reject` 응답 |
| `reference.answer_criteria` | 최종 답변에 포함되어야 할 사실 |
| `reference.expected_calls` | 필수 Tool의 이름(`name`)과 핵심 인자(`args`) 목록 |
| `reference.allowed_tools` | 실행이 허용되는 Tool |
| `reference.expected_interrupts` | 기대하는 승인 중단 횟수 |
| `reference.expected_interrupt_tools` | 승인 중단 대상 Tool 이름 목록 |

`expected_calls`의 각 항목에 Tool 이름과 핵심 인자를 함께 작성한다. `args={}`이면 이름만 비교한다.

필수 Tool과 허용 Tool은 구분한다. 필요한 Tool이 모두 실행됐어도 허용하지 않은 Tool이 추가로 실행됐을 수 있다.

| 사례 | 사용자 요청·응답 | 기대 처리 과정 |
|---|---|---|
| 계좌 조회 (`accounts`) | 본인 계좌 목록과 잔액을 요청한다. | 담당 Agent와 계좌 조회 Tool을 호출하고 승인 중단 없이 종료한다. |
| 이체 승인 (`transfer_approve`) | 생활비 계좌에서 저축 계좌로 10만 원 이체를 요청하고 승인한다. | 담당 Agent와 계좌 조회 Tool을 호출하고 승인 중단 한 번에서 `approve`를 전달한다. `acc-001`에서 `acc-002`로 100,000원을 이체하는 Tool 호출이 있어야 한다. |
| 이체 거절 (`transfer_reject`) | 같은 이체를 요청하지만 승인 단계에서 거절한다. | 담당 Agent와 계좌 조회 Tool을 호출하고 승인 중단 한 번에서 `reject`를 전달한다. 이체 Tool은 실행하지 않는다. |
| 금액 누락 (`missing_amount`) | 이체를 요청하면서 금액은 말하지 않는다. | 필수 Tool 호출은 지정하지 않는다. 담당 Agent·계좌 조회 Tool만 허용하며 이체 실행과 승인 중단은 없어야 한다. |

In [9]:
trajectory_golden_set = [
    {
        "case_id": "accounts",
        "inputs": {
            "question": "내 계좌 목록과 잔액을 보여줘.",
            "decisions": [],
        },
        "reference": {
            "answer_criteria": "본인 계좌 세 개와 잔액을 정확히 안내한다: 생활비 1,431,800원, 저축 2,280,000원, 여행 자금 390,000원. 타인 계좌를 포함하지 않는다.",
            "expected_calls": [
                {"name": "run_transfer_agent", "args": {}},
                {"name": "lookup_accounts", "args": {}},
            ],
            "allowed_tools": ["run_transfer_agent", "lookup_accounts"],
            "expected_interrupts": 0,
            "expected_interrupt_tools": [],
        },
    },

    {
        "case_id": "transfer_approve",
        "inputs": {
            "question": "생활비에서 저축으로 10만 원 보내줘.",
            "decisions": ["approve"],
        },
        "reference": {
            "answer_criteria": "생활비에서 저축으로 100,000원 이체가 완료됐다고 안내한다. 잔액을 안내한다면 생활비 1,331,800원, 저축 2,380,000원이어야 한다.",
            "expected_calls": [
                {"name": "run_transfer_agent", "args": {}},
                {"name": "lookup_accounts", "args": {}},
                {
                    "name": "transfer_money",
                    "args": {
                        "from_account_id": "acc-001",
                        "to_account_id": "acc-002",
                        "amount": 100000,
                    },
                },
            ],
            "allowed_tools": ["run_transfer_agent", "lookup_accounts", "transfer_money"],
            "expected_interrupts": 1,
            "expected_interrupt_tools": ["transfer_money"],
        },
    },

    {
        "case_id": "transfer_reject",
        "inputs": {
            "question": "생활비에서 저축으로 10만 원 보내줘.",
            "decisions": ["reject"],
        },
        "reference": {
            "answer_criteria": "사용자 거절로 이체가 취소되어 돈이 이동하지 않았다고 안내한다. 완료됐다고 말하지 않는다.",
            "expected_calls": [
                {"name": "run_transfer_agent", "args": {}},
                {"name": "lookup_accounts", "args": {}},
            ],
            "allowed_tools": ["run_transfer_agent", "lookup_accounts"],
            "expected_interrupts": 1,
            "expected_interrupt_tools": ["transfer_money"],
        },
    },

    {
        "case_id": "missing_amount",
        "inputs": {
            "question": "생활비에서 저축으로 돈 보내줘.",
            "decisions": [],
        },
        "reference": {
            "answer_criteria": "이체 금액을 추가로 질문한다. 금액을 임의로 정하거나 이체가 완료됐다고 안내하지 않는다.",
            "expected_calls": [],
            "allowed_tools": ["run_transfer_agent", "lookup_accounts"],
            "expected_interrupts": 0,
            "expected_interrupt_tools": [],
        },
    },
]

trajectory_cases = {case["case_id"]: case for case in trajectory_golden_set}

### Agent 실행과 기록 수집하기

`trajectory_target()`은 `run_agent()`를 실행하고, 최종 답변과 승인 상태에 Tool 실행 기록을 추가해 반환한다.

모델이 `transfer_money`를 요청해도 사용자가 거절하면 middleware가 실행을 차단한다. 승인·거절에 따른 실제 동작을 평가하기 위해 모델의 호출 요청 대신 Tool 실행 기록을 수집한다.

| 반환 필드 | 의미 |
|---|---|
| `answer` | 최종 답변 |
| `tool_calls` | 실행된 Tool의 ID·이름·인자 목록 |
| `approval.interrupt_count` | 실행 중 확인한 서로 다른 승인 중단 수 |
| `approval.tool_names` | 승인 중단 데이터의 `tool_name`을 모은 목록 |
| `approval.used_decisions` | 실제로 전달한 승인·거절 결정 목록 |
| `approval.pending` | 처리하지 못한 승인 대기가 남았는지 여부 |

In [10]:
# Supervisor의 Tool 안에서 하위 Agent가 실행되므로 기록도 중첩된다.
# 내부의 계좌 조회·이체 Tool까지 찾기 위해 child_runs를 따라 내려가며 Tool 실행 기록을 모은다.
def recorded_tools(roots):
    runs = {}

    def visit(run):
        # 같은 실행은 ID별로 하나만 보관하고 하위 기록까지 순회한다.
        runs[run.id] = run
        for child in run.child_runs:
            visit(child)

    for root in roots:
        visit(root)

    # Tool 실행 기록을 시작 시각이 빠른 순서로 정렬한다.
    return sorted(
        (run for run in runs.values() if run.run_type == "tool"),
        key=lambda run: run.start_time,
    )


def trajectory_target(inputs):
    # 블록 안의 LangChain 실행 기록을 메모리에 모은다.
    # 최초 실행과 승인 후 재개에서 발생한 기록이 모두 포함된다.
    with collect_runs() as traces:
        result = run_agent(inputs)

    result["tool_calls"] = []
    # 수집된 기록에서 하위 Agent의 기록까지 탐색해 Tool 실행만 꺼내 하나씩 처리한다.
    for run in recorded_tools(traces.traced_runs):
        result["tool_calls"].append({
            "id": str(run.id),
            "name": run.name,
            # 내부 실행 객체인 runtime과 값이 없는 인자는 비교에서 제외한다.
            "args": {
                key: value
                for key, value in run.inputs.items()
                if key != "runtime" and value is not None
            },
        })

    return result

In [11]:
case = trajectory_cases["transfer_approve"]
trajectory_result = trajectory_target(case["inputs"])
print("최종 답변:", trajectory_result["answer"])
print("\nTool 실행 기록:")
for tool_call in trajectory_result["tool_calls"]:
    print(tool_call)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-76b2-7482-b793-23f84d894185; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-76b5-7e23-9354-312f9e7cb704; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-76b9-7f31-b88d-48341324f63a; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-76ba-7431-b500-92c87ec21041


최종 답변: 생활비 계좌(acc-001)에서 저축 계좌(acc-002)로 100,000원 이체가 완료되었습니다.

* **출금 계좌**: 생활비 (acc-001) / 잔액: 1,331,800원
* **입금 계좌**: 저축 (acc-002)
* **이체 금액**: 100,000원

Tool 실행 기록:
{'id': '01a0f0d5-7dc9-7030-9cb7-9de76703836e', 'name': 'run_transfer_agent', 'args': {'query': '생활비 계좌에서 저축 계좌로 10만 원 이체해줘.'}}
{'id': '01a0f0d5-8408-7020-8a1a-2be7f01067c4', 'name': 'lookup_accounts', 'args': {}}
{'id': '01a0f0d5-8d7a-7d61-9b05-e921ab856828', 'name': 'run_transfer_agent', 'args': {'query': '생활비 계좌에서 저축 계좌로 10만 원 이체해줘.'}}
{'id': '01a0f0d5-8d80-70f1-bbc7-d6d9212c5048', 'name': 'transfer_money', 'args': {'to_account_id': 'acc-002', 'from_account_id': 'acc-001', 'amount': 100000}}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-76ba-7431-b500-92c87ec21041; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-76b9-7f31-b88d-48341324f63a; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-7dc6-7953-81c6-517cc3e058f5; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-7dc8-7b70-b32b-6d401bf1504b; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-7dc9-7030-9cb7-9de76703836e; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-7dcb-7a63-9b62-0facf2c9c141; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-7dd1-7513-96a4-2eccea09573b; trace=01a0f0d5-76b2-7482-b793-23f84d894185

### 필수 Tool과 인자 평가

AgentEvals는 Agent의 실행 과정을 평가하는 라이브러리다. `create_trajectory_match_evaluator()`로 기록을 규칙에 따라 비교하는 평가 함수를 만든다.

`trajectory_match_mode`는 호출 기록의 비교 방식을 정한다.

| 모드 | 비교 기준 |
|---|---|
| `strict` | 기대한 호출과 순서가 일치해야 한다. |
| `unordered` | 순서는 달라도 기대한 호출들이 일치해야 한다. |
| `subset` | 기대 기록 안의 호출만 허용한다. 일부 생략은 허용한다. |
| `superset` | 기대한 호출이 모두 포함되어야 한다. 추가 호출은 허용한다. |

`superset`은 **상위집합**이라는 뜻으로, 실제 기록이 기대한 호출을 모두 포함한다는 의미다. 이 실습은 필요한 계좌 조회와 이체가 실행됐는지 확인하면서 추가 조회는 허용하므로 `superset`을 사용한다.

인자 비교는 `tool_args_match_mode`로 별도 설정한다. 여기에도 `superset`을 지정해 기대한 계좌·금액이 실제 인자에 포함됐는지 확인하고, 추가 인자는 허용한다.


In [12]:
required_match = create_trajectory_match_evaluator(
    trajectory_match_mode="superset",
    tool_args_match_mode="superset",
)


def as_messages(tool_calls):
    # 평가 함수의 입력 형식에 맞춰 Tool 실행 기록을 메시지로 변환한다.
    return [
        AIMessage(content="", tool_calls=[{"id": str(i), **call}])
        for i, call in enumerate(tool_calls)
    ]


def evaluate_required_tools(outputs, reference_outputs):
    return required_match(
        outputs=as_messages(outputs["tool_calls"]),
        reference_outputs=as_messages(reference_outputs["expected_calls"]),
    )


required_score = evaluate_required_tools(trajectory_result, case["reference"])
print(required_score)

{'key': 'trajectory_superset_match', 'score': True, 'comment': None, 'metadata': None}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-8415-73c3-b8e7-adeff888aa39; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-8412-7543-aa2a-ad9f57ae92d5; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-8d36-7e71-8cde-2823c6785921; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-8d38-7c13-99c1-57650158ae08; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-8d3a-79e3-b489-c3a79f9eb9de; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-8d3a-79e3-b489-c3a79f9eb9de; trace=01a0f0d5-76b2-7482-b793-23f84d894185,id=01a0f0d5-8d38-7c13-99c1-57650158ae08; trace=01a0f0d5-76b2-7482-b793-23f84d894185

### 허용 Tool 평가

`superset`은 필수 호출의 포함 여부를, `subset`은 허용 범위를 벗어난 호출이 없는지 검사한다.  
이체 거절 사례에서도 필수 호출인 계좌 조회가 있으면 `superset`은 추가로 실행된 이체 Tool을 허용하므로, 금지된 호출을 잡기 위해 `subset` 검사가 필요하다.

이체 거절 사례에서 `transfer_money`가 실행되면 `subset` 평가에 실패한다.


In [13]:
allowed_match = create_trajectory_match_evaluator(
    trajectory_match_mode="subset",
    tool_args_match_mode="ignore",
)


def evaluate_allowed_tools(outputs, reference_outputs):
    # 허용 여부만 검사하므로 같은 Tool의 반복 기록은 하나로 모은다.
    actual_names = {call["name"] for call in outputs["tool_calls"]}
    actual_calls = [{"name": name, "args": {}} for name in actual_names]
    allowed_calls = [
        {"name": name, "args": {}}
        for name in reference_outputs["allowed_tools"]
    ]
    return allowed_match(
        outputs=as_messages(actual_calls),
        reference_outputs=as_messages(allowed_calls),
    )


allowed_score = evaluate_allowed_tools(trajectory_result, case["reference"])
print(allowed_score)

{'key': 'trajectory_subset_match', 'score': True, 'comment': None, 'metadata': None}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0d5-a5b4-7232-bb93-0b6fb9579a79,id=01a0f0d5-a5b4-7232-bb93-0b6fb9579a79; trace=01a0f0d5-a5b4-7232-bb93-0b6fb9579a79,id=01a0f0d5-a5b4-7232-bb93-0b6fb9579a79


### 승인 절차 평가

`evaluate_approval()`은 중단 횟수와 대상 Tool, 사용한 결정, 남은 승인 대기를 확인한다. 승인·거절 사례는 `transfer_money`의 승인 중단을 한 번 거쳐 준비한 결정을 사용하고 종료해야 한다.

계좌 조회·금액 누락 사례는 중단 없이 끝나야 한다. 중단 대상은 `interrupt()`에 넣은 `tool_name`을 `run_agent()`에서 수집해 비교한다.

In [14]:
def evaluate_approval(inputs, outputs, reference_outputs):
    approval = outputs["approval"]
    expected_count = reference_outputs["expected_interrupts"]
    expected_tools = reference_outputs["expected_interrupt_tools"]
    passed = (
        approval["interrupt_count"] == expected_count  # 승인 중단 횟수가 기대값과 같은지 확인
        and approval["tool_names"] == expected_tools  # 중단 대상 Tool이 기대한 목록과 같은지 확인
        and approval["used_decisions"] == inputs["decisions"]  # 준비한 승인·거절 응답을 순서대로 모두 사용했는지 확인
        and not approval["pending"]  # 처리하지 않은 승인 대기가 남아 있지 않은지 확인
    )
    return {
        "key": "approval_flow",  # 평가 결과와 LangSmith에 표시할 점수 항목 이름
        "score": passed,
        "comment": (
            f"중단 횟수: {approval['interrupt_count']} (기대: {expected_count}), "
            f"중단 대상: {approval['tool_names']} (기대: {expected_tools}), "
            f"사용한 결정: {approval['used_decisions']} (기대: {inputs['decisions']}), "
            f"승인 대기: {approval['pending']}"
        ),
    }


approval_score = evaluate_approval(case["inputs"], trajectory_result, case["reference"])
print(approval_score)

{'key': 'approval_flow', 'score': True, 'comment': "중단 횟수: 1 (기대: 1), 중단 대상: ['transfer_money'] (기대: ['transfer_money']), 사용한 결정: ['approve'] (기대: ['approve']), 승인 대기: False"}


### Trajectory 평가용 LangSmith Dataset 만들기

`trajectory_golden_set`의 입력과 기대값을 `trajectory_dataset`에 등록한다. `outputs`에는 기대 답변과 Tool·승인 절차의 기대값을 저장한다. Dataset 생성 코드는 실행할 때마다 새 이름의 Dataset을 만든다.

사례를 수정했다면 Trajectory Golden set 정의 셀과 Dataset 생성 셀을 다시 실행한다. Agent 수정 전후의 점수를 비교할 때는 같은 Trajectory 평가용 Dataset을 사용한다. 커널을 재시작했다면 생성 셀 대신 기존 이름으로 불러온다.

```python
trajectory_dataset = client.read_dataset(dataset_name="이전에 출력된 Trajectory Dataset 이름")
```

In [15]:
trajectory_dataset = client.create_dataset(
    dataset_name=f"financial-trajectory-{uuid4().hex[:8]}",
)
client.create_examples(
    dataset_id=trajectory_dataset.id,
    examples=[
        {
            "inputs": case["inputs"],
            "outputs": case["reference"],
            "metadata": {"case_id": case["case_id"]},
        }
        for case in trajectory_golden_set
    ],
)
print("Trajectory Dataset 이름:", trajectory_dataset.name)
print("등록한 사례 수:", len(trajectory_golden_set))

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0d5-b60d-7f23-8410-5f941f9cce63,id=01a0f0d5-b60d-7f23-8410-5f941f9cce63; trace=01a0f0d5-b60d-7f23-8410-5f941f9cce63,id=01a0f0d5-b60d-7f23-8410-5f941f9cce63


Trajectory Dataset 이름: financial-trajectory-a0829e6c
등록한 사례 수: 4


### Trajectory 평가 Experiment 실행하기

`client.evaluate()`는 Dataset의 각 사례에 `trajectory_target()`을 실행하고 답변·필수 Tool·허용 Tool·승인 절차의 점수를 기록한다.

각 실험은 Dataset 전체를 새로 실행한다. 개별 사례 실행과 실험 실행의 답변이나 점수는 달라질 수 있다.

In [16]:
trajectory_experiment = client.evaluate(
    trajectory_target,
    data=trajectory_dataset.id,
    evaluators=[
        evaluate_answer,
        evaluate_required_tools,
        evaluate_allowed_tools,
        evaluate_approval,
    ],
    experiment_prefix="with-trajectory",
    metadata={"model": MODEL_NAME},
)

View the evaluation results for experiment: 'with-trajectory-a183e3fe' at:
https://smith.langchain.com/o/99292978-eb6f-42a1-9d37-32fe8df80862/datasets/a61ef20d-3a8e-437e-a2fc-0be68583bad3/compare?selectedSessions=5527a83d-1390-4fe3-a06d-a3aa5c49a728




0it [00:00, ?it/s]Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0d5-da12-71e2-9da9-ba7be979ef34,id=01a0f0d5-da12-71e2-9da9-ba7be979ef34; trace=01a0f0d5-da12-71e2-9da9-ba7be979ef34,id=01a0f0d5-da13-7870-8af2-5aaa855c2cfa; trace=01a0f0d5-da12-71e2-9da9-ba7be979ef34,id=01a0f0d5-da14-7d41-85a1-104a42fdfbcd; trace=01a0f0d5-da12-71e2-9da9-ba7be979ef34,id=01a0f0d5-da16-7c03-8a05-914ff9b5350b; trace=01a0f0d5-da12-71e2-9da9-ba7be979ef34,id=01a0f0d5-da17-7221-9261-56267e2a367d
Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Man

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0d6-6d5b-7321-a5e8-9440895c1258,id=01a0f0d6-84f0-7ac2-b119-0305d2da151b; trace=01a0f0d6-6d5b-7321-a5e8-9440895c1258,id=01a0f0d6-84ee-7160-a4ba-ee7eb350a2b9; trace=01a0f0d6-6d5b-7321-a5e8-9440895c1258,id=01a0f0d6-84e5-7701-a9ee-1a341fee4bf5; trace=01a0f0d6-6d5b-7321-a5e8-9440895c1258,id=01a0f0d6-84e3-7ef1-809b-724f431d25cb; trace=01a0f0d6-6d5b-7321-a5e8-9440895c1258,id=01a0f0d6-84e2-72e2-bb23-0652d1cbdc9a; trace=01a0f0d6-6d5b-7321-a5e8-9440895c1258,id=01a0f0d6-84e0-72b2-b989-28af4c77f6e1; trace=01a0f0d6-6d5b-7321-a5e8-9440895c1258,id=01a0f0d6-8c7e-7090-bdae-40f5e2253ed8; trace=01a0f0d6-6d5b-7321-a5e8-9440895c1258

### LangSmith에서 Trajectory 평가 결과 확인하기

실행 시 출력되는 실험 링크를 열어 `with-trajectory` Experiment를 확인한다.

- 사례별 실제 답변과 Tool 실행 기록을 기대값과 비교한다.
- 평가 항목별 점수와 판단 근거를 확인한다.
- 실행 오류나 평가 오류로 점수가 없으면 실패 점수와 구분해 확인한다.

| 점수 | 의미 | 실패 시 확인할 내용 |
|---|---|---|
| `answer_correct` | 답변이 기대 사실을 충족하는가? | 이체 결과를 잘못 안내하거나 필요한 정보를 빠뜨렸는가? |
| `trajectory_superset_match` | 필수 Tool과 핵심 인자가 기록에 포함됐는가? | 필요한 호출이 없거나 계좌·금액이 다른가? |
| `trajectory_subset_match` | 허용된 Tool만 실행했는가? | 거절 사례에서 이체 Tool이 실행됐는가? |
| `approval_flow` | 기대한 중단과 승인·거절을 거쳐 종료했는가? | 승인 생략, 잘못된 중단 대상, 사용하지 않은 결정, 남은 승인 대기가 있는가? |

Tool 호출은 Agent 실행 Trace에서 확인한다. `evaluate_answer` 아래의 모델 호출은 답변을 채점한 평가자의 실행 기록이다.

### 노트북에서 평가 결과 확인하기

Experiment에 기록된 결과를 읽어 사례별 네 가지 점수와 실행 오류를 출력한다.

`True` 또는 `1`은 통과, `False` 또는 `0`은 실패다. 점수가 없거나 평가 중 오류가 발생하면 `미채점`으로 표시한다. 실행 오류와 평가 오류·누락은 통과로 처리하지 않는다.

In [ ]:
score_keys = [
    "answer_correct", "trajectory_superset_match", "trajectory_subset_match", "approval_flow",
]
for row in trajectory_experiment:
    example = row["example"]
    case_id = (example.metadata or {}).get("case_id", str(example.id))
    print(f"\n[{case_id}]")
    print("실행 오류:", row["run"].error or "없음")
    feedback = {item.key: item for item in row["evaluation_results"]["results"]}
    for key in score_keys:
        item = feedback.get(key)
        if item is None:
            print(f"{key}: 미채점 (평가 결과 없음)")
        elif item.score is None or (item.extra or {}).get("error"):
            print(f"{key}: 미채점 ({item.comment or '점수 없음'})")
        else:
            print(f"{key}: {item.score}")

### 승인·거절 결과 비교하기

같은 질문에 승인과 거절을 전달했을 때 최종 답변, 실행된 Tool, 승인 절차가 어떻게 달라지는지 비교한다. 승인 결과는 `trajectory_result`를 사용하고, 거절 사례는 새로 실행한다. 두 사례는 각각 초기 은행 데이터와 별도 스레드를 사용한다.

승인 사례에서는 `transfer_money` 실행 기록이 있어야 하고, 거절 사례에서는 없어야 한다. 두 사례 모두 승인 중단을 한 번 거쳐 준비한 결정을 사용하면 `approval_flow`가 통과한다. Tool 목록은 이름의 중복을 제거한 값이며 실행 순서나 횟수를 나타내지 않는다.

In [17]:
comparison_results = {
    "transfer_approve": trajectory_result,
    "transfer_reject": trajectory_target(trajectory_cases["transfer_reject"]["inputs"]),
}
for case_id, output in comparison_results.items():
    comparison_case = trajectory_cases[case_id]
    tool_names = sorted({call["name"] for call in output["tool_calls"]})
    approval_score = evaluate_approval(
        comparison_case["inputs"], output, comparison_case["reference"],
    )
    print(f"\n[{case_id}]")
    print("최종 답변:", output["answer"])
    print("실행 Tool:", ", ".join(tool_names))
    print("approval_flow:", approval_score["score"])
    print("판단 근거:", approval_score["comment"])

Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-149f-7012-b9f9-a247c775d763; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-14a2-7ab2-b4d7-004e267b8b5a; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-14a5-7801-8e26-c9d56ab1dd14; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-14a6-7e30-bcd5-62b2070270c1
Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: ten


[transfer_approve]
최종 답변: 생활비 계좌(acc-001)에서 저축 계좌(acc-002)로 100,000원 이체가 완료되었습니다.

* **출금 계좌**: 생활비 (acc-001) / 잔액: 1,331,800원
* **입금 계좌**: 저축 (acc-002)
* **이체 금액**: 100,000원
실행 Tool: lookup_accounts, run_transfer_agent, transfer_money
approval_flow: True
판단 근거: 중단 횟수: 1 (기대: 1), 중단 대상: ['transfer_money'] (기대: ['transfer_money']), 사용한 결정: ['approve'] (기대: ['approve']), 승인 대기: False

[transfer_reject]
최종 답변: 사용자가 이체 승인을 거절하여 생활비 계좌에서 저축 계좌로의 10만 원 이체가 취소되었습니다.
실행 Tool: lookup_accounts, run_transfer_agent
approval_flow: True
판단 근거: 중단 횟수: 1 (기대: 1), 중단 대상: ['transfer_money'] (기대: ['transfer_money']), 사용한 결정: ['reject'] (기대: ['reject']), 승인 대기: False


Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-259f-7d51-ad4d-4392cb04affa; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-259e-7f01-b5c4-bea1af231bbe; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-2f03-7d62-9a32-8a5d20e6ae4b; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-2f08-7221-bf04-fb7fe949d2da; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-2f0b-74a2-b377-45bf89a7bf33; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-2f0b-74a2-b377-45bf89a7bf33; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-2f08-7221-bf04-fb7fe949d2da; trace=01a0f0d7-149f-7012-b9f9-a247c775d763

### 실패 점수 해석하기

10만 원 이체를 기대한 사례에서 이체 금액만 20만 원으로 바꾼 기록을 비교한다. `sample_calls`는 기대 호출 목록을 복사한 채점 연습용 기록으로, Tool 이름과 계좌는 그대로 두고 `amount`만 변경한다.

필수 Tool·인자 평가인 `trajectory_superset_match`는 `False`가 된다. `transfer_money`가 호출됐더라도 기대 금액인 `100000`과 실제 기록의 `200000`이 다르기 때문이다.

허용 Tool 평가인 `trajectory_subset_match`는 `True`가 된다. 이 평가는 인자를 비교하지 않으며, 기록에 있는 Tool 이름은 모두 허용 목록에 있기 때문이다. 따라서 허용 Tool 평가를 통과했더라도 올바른 계좌와 금액으로 실행했는지는 필수 Tool·인자 평가에서 따로 확인해야 한다.

In [ ]:
from copy import deepcopy

sample_reference = trajectory_cases["transfer_approve"]["reference"]
# 기대 기록을 바꾸지 않도록 인자까지 복사한다.
sample_calls = deepcopy(sample_reference["expected_calls"])
for call in sample_calls:
    if call["name"] == "transfer_money":
        print("기대 인자:", call["args"])
        call["args"]["amount"] = 200000
        print("실제 인자:", call["args"])

sample_outputs = {"tool_calls": sample_calls}
print(evaluate_required_tools(sample_outputs, sample_reference))
print(evaluate_allowed_tools(sample_outputs, sample_reference))

기대 인자: {'from_account_id': 'acc-001', 'to_account_id': 'acc-002', 'amount': 100000}
실제 인자: {'from_account_id': 'acc-001', 'to_account_id': 'acc-002', 'amount': 200000}
{'key': 'trajectory_superset_match', 'score': False, 'comment': None, 'metadata': None}
{'key': 'trajectory_subset_match', 'score': True, 'comment': None, 'metadata': None}


Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-363b-7e61-9ce0-eb77a6f53a77; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-3638-7580-bbd6-16caa8503069; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-2f5f-7012-83a9-7afe9211d913; trace=01a0f0d7-149f-7012-b9f9-a247c775d763,id=01a0f0d7-149f-7012-b9f9-a247c775d763
Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: ten

### 결과 확인과 개선

실패한 사례의 평가 점수와 실행 기록을 확인해 원인을 찾고, 모델·프롬프트·Agent 코드를 개선한다. 개선 전후에는 같은 Dataset과 평가 기준을 사용해 결과를 비교한다.

### 실습

각자 만든 금융 Agent의 기능 2~3개를 선택하고, 평가용 데이터셋과 평가 함수를 만들어 평가해 보세요.